### Change Data Capture (CDC) from an API into a database

In [ ]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import json 
import psycopg2
import os
import logging
# Set the option to display all columns
pd.set_option('display.max_columns',100)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

In [ ]:
"""Config stage: load API and database settings from the local .env file."""
def load_env_file(env_path=".env"):
    """Load key value pairs from the local .env file."""
    print(f"[config] loading settings from {env_path}")
    logging.info(f"config load started from {env_path}")
    with open(env_path, "r") as env_file:
        for line in env_file:
            clean_line = line.strip()
            if not clean_line or clean_line.startswith("#"):
                continue
            key, value = clean_line.split("=", 1)
            os.environ[key] = value

load_env_file()
url = os.environ["API_URL"]
db_username = os.environ["DB_USERNAME"]
db_password = os.environ["DB_PASSWORD"]
db_host = os.environ["DB_HOST"]
db_port = os.environ["DB_PORT"]
db_name = os.environ["DB_NAME"]
table_name = os.environ["TABLE_NAME"]
print(f"[config] database={db_name}, host={db_host}, table={table_name}")
logging.info(f"config loaded for database {db_name} on {db_host}")

In [45]:
"""Extract data from a Public API """
def extract_data(url):
    """Extract data from a Public API """
    print(f"Extracting data from {url}...")
    try:
        response = requests.get(url).json()
        return response
    except FileNotFoundError:
        print(f"Error: {url} not found!!")
        return None


In [46]:
"""Transform the data (1)"""
def transform_data(response):
    """ Data Parsing: Parse the API response (e.g., JSON, XML) and extract the necessary fields. If the API data structure differs from your target database schema, perform any necessary transformations (renaming columns, converting data types)"""
    #if response is None:
    #    return None
    print("Transforming data...")
    data= response['Time Series (Daily)']
    dates=data.keys()
    info= data.values()
    # trurn to pandas Dataframes 
    info= pd.DataFrame(info)
    dates=pd.DataFrame(dates)
    # Add last updated column 
    last_updated = response['Meta Data']['3. Last Refreshed']
    # turn to pandas Dataframe
    last_updated = pd.DataFrame([last_updated], columns=['Last_updated'])
    # Merge all Dataframes USING concat function
    result = pd.concat([dates, info,last_updated], axis=1)
    result=pd.DataFrame(result)
    #result['Date']=pd.to_datetime(result['Date'])
    # Fill NaN with ffill method 
    result['Last_updated']= result['Last_updated'].fillna(method='ffill')
    # rename columns
    result=result.rename(columns={0:'Date',
                                  '1. open':'Open',
                                  '2. high':'High',
                                  '3. low': 'Low',
                                  '4. close':'Close',
                                  '5. volume':'Volume'}, 
                         inplace=True)   
    return result
   

In [47]:
"""Connect to the target database-- warehouse"""
def create_connection(db_username, db_password,db_host,db_port,db_name):
    # Create a database engine
    engine = create_engine(f'postgresql://{db_username}:{db_password}@{db_host}:         {db_port}/{db_name}', echo=False)
    conn = psycopg2.connect(f'postgresql://{db_username}:{db_password}@{db_host}:     {db_port}/{db_name}')
    #cur = conn.cursor()
    print("Connecting to Target data Warhouse...")
    return conn, engine

In [48]:
"""Read data from the target table into dataframe"""
def  high_water_mark (conn,table_name):
    
    print(f"Reading Data from {table_name} table")
   
   # Corrected: Use read_sql_table with the engine directly
    target = pd.read_sql_table(table_name, con=conn)
    
    # Calculate the max timestamp
    last_timestamp = target['Date'].max()
    
    return last_timestamp


In [49]:
""" Transform data (2) """
def new_data_df(result, last_timestamp):
    print(f"Comparing Data after {last_timestamp}")
    result['Date'] = pd.to_datetime(result['Date'], errors='coerce') 
    last_timestamp = pd.to_datetime(last_timestamp, errors='coerce')
    new_data = result[result['Date'] > last_timestamp]
    return new_data 

Insert new data to staging database : BMI_history

In [50]:
"""Load the new data into an stage table."""
def load_data(new_data,engine ,table_name):
    if new_data is None:
        return None 
    new_data=pd.Dataframe(new_data)
    print(f"Loading data into {table_name}...")                     
    new_data.to_sql(table_name, engine, if_exists='append', index=False, chunksize=100)
    print("update operation completed successfully.")


In [51]:
def run_etl_pipeline():
    """Orchestrates the ETL process."""
    print("ETL Pipeline Started...")
    
    # 1. Define missing configurations (usually from a config file or env vars)
    #url = "https://api.example.com/data"
    #table_name = "target_table"
    # db_username, db_password, etc., should be defined here or globally

    # E: Extract
    response = extract_data(url)
    
    # T: Transform 
    result = transform_data(response)
    
    # Create connection components
    conn, engine = create_connection(db_username, db_password, db_host, db_port, db_name)
    
    # Important: Pass 'engine' to high_water_mark if it uses pd.read_sql_table
    last_timestamp = high_water_mark(engine, table_name)
    
    # Filter for new records
    new_data = new_data_df(result, last_timestamp)
    
    # L: Load
    if not new_data.empty:
        load_data(new_data, engine, table_name)
        print(f"Loaded {len(new_data)} new records.")
    else:
        print("No new data to load.")
        
    # Close resources
    conn.close()
    print("ETL Pipeline Completed.")


    run_etl_pipeline()



#### 4. Silver layer refresh
This stage cleans the staged history table and keeps the latest record for each business date before loading the silver table.

In [ ]:
from pathlib import Path

"""Setup stage: load the SQL file, show the statements, and create the silver and gold objects."""
def execute_sql_text(sql_text, connection):
    """Show the SQL before executing it and log the stage progress."""
    print(sql_text)
    logging.info("running sql block")
    with connection.cursor() as cursor:
        cursor.execute(sql_text)
    connection.commit()
    logging.info("sql block completed")

sql_file_path = Path("IBM_StoredProcedure.sql")
sql_setup = sql_file_path.read_text()

conn = psycopg2.connect(
    f"postgresql://{db_username}:{db_password}@{db_host}:{db_port}/{db_name}"
)
execute_sql_text(sql_setup, conn)
print("[setup] silver and gold objects are ready")

#### 5. Gold layer refresh
This stage runs the silver procedure first, then calculates rolling metrics for the gold layer.

In [ ]:
"""Silver stage: refresh the clean latest-per-day layer from the staging table."""
silver_query = "CALL refresh_ibm_silver();"
execute_sql_text(silver_query, conn)
print("[silver] refresh completed")
logging.info("silver refresh completed")

"""Gold stage: build analytics-ready measures from the silver layer."""
gold_query = "CALL refresh_ibm_gold();"
execute_sql_text(gold_query, conn)
print("[gold] refresh completed")
logging.info("gold refresh completed")

#### 6. Verification queries
Review the SQL first, then confirm the silver and gold row counts and inspect the latest gold rows.

In [ ]:
"""Verification stage: check the silver and gold tables after the refresh steps."""
silver_check_query = '''
SELECT count(*) AS silver_rows, min("Date") AS silver_min_date, max("Date") AS silver_max_date
FROM public."IBM";
'''

gold_check_query = '''
SELECT count(*) AS gold_rows, min("Date") AS gold_min_date, max("Date") AS gold_max_date
FROM public."IBM_gold";
'''

gold_sample_query = '''
SELECT *
FROM public."IBM_gold"
ORDER BY "Date" DESC
LIMIT 10;
'''

print(silver_check_query)
silver_check = pd.read_sql(silver_check_query, conn)
print(silver_check)

print(gold_check_query)
gold_check = pd.read_sql(gold_check_query, conn)
print(gold_check)

print(gold_sample_query)
gold_sample = pd.read_sql(gold_sample_query, conn)
gold_sample